# Building a Simple Word-Level Tokenizer

This notebook demonstrates how to construct a small word-level tokenizer from a local text corpus.

It uses the repository dataset:

```text
NLP/datasets/alice_in_wonderland.txt
```

No PDF extraction, package-installation cells, external downloads, or runtime dataset downloads are required.

## Learning objectives

After completing this notebook, students should be able to:

1. Load and inspect a local text corpus.
2. Normalize text consistently.
3. Split text into sentences and tokens.
4. Build a deterministic vocabulary.
5. Reserve IDs for special tokens.
6. Encode text as token IDs.
7. Decode token IDs back to text.
8. Handle unknown words, truncation, and padding.
9. Wrap the complete workflow in a reusable tokenizer class.

## 1. Import the required libraries

In [ ]:
from pathlib import Path
import re
import unicodedata

import torch

## 2. Load the local corpus

The notebook is located in `NLP/notebooks`, so the dataset is referenced through a portable relative path.

In [ ]:
data_path = Path("../datasets/alice_in_wonderland.txt")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {data_path.resolve()}"
    )

corpus_text = data_path.read_text(
    encoding="utf-8",
    errors="replace",
)

print("Characters:", len(corpus_text))
print("Preview:")
print(corpus_text[:500])

## 3. Normalize the text

The cleaning function converts Unicode text to a consistent form, converts text to lowercase, keeps letters and selected punctuation, and collapses repeated whitespace.

In [ ]:
def normalize_text(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    ascii_text = normalized.encode(
        "ascii",
        "ignore",
    ).decode("ascii")

    ascii_text = ascii_text.lower()
    ascii_text = re.sub(
        r"[^a-z0-9'.,!?;:\s-]+",
        " ",
        ascii_text,
    )
    ascii_text = re.sub(r"\s+", " ", ascii_text)

    return ascii_text.strip()


cleaned_text = normalize_text(corpus_text)

print("Cleaned characters:", len(cleaned_text))
print("Preview:")
print(cleaned_text[:500])

## 4. Split the corpus into sentences

This simple rule-based splitter is intentionally lightweight. Production systems should use a language-aware sentence segmenter.

In [ ]:
def split_sentences(text: str) -> list[str]:
    parts = re.split(r"(?<=[.!?])\s+", text)

    return [
        sentence.strip()
        for sentence in parts
        if sentence.strip()
    ]


sentences = split_sentences(cleaned_text)

print("Sentence count:", len(sentences))
print("\nFirst three sentences:")
for sentence in sentences[:3]:
    print("-", sentence)

## 5. Define the tokenization rule

The tokenizer separates words and selected punctuation marks.

In [ ]:
TOKEN_PATTERN = re.compile(
    r"[a-z0-9]+(?:'[a-z0-9]+)?|[.,!?;:]"
)


def tokenize(text: str) -> list[str]:
    normalized = normalize_text(text)
    return TOKEN_PATTERN.findall(normalized)


example_text = "Alice's adventures are curious, aren't they?"
example_tokens = tokenize(example_text)

print(example_tokens)

## 6. Build a deterministic vocabulary

Special tokens are placed first so their IDs remain stable:

- `[PAD]`: padding token;
- `[UNK]`: unknown token;
- `[BOS]`: beginning-of-sequence token;
- `[EOS]`: end-of-sequence token.

In [ ]:
SPECIAL_TOKENS = [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]

corpus_tokens = tokenize(cleaned_text)
lexical_tokens = sorted(set(corpus_tokens))

vocabulary = SPECIAL_TOKENS + [
    token
    for token in lexical_tokens
    if token not in SPECIAL_TOKENS
]

token_to_id = {
    token: index
    for index, token in enumerate(vocabulary)
}

id_to_token = {
    index: token
    for token, index in token_to_id.items()
}

print("Corpus tokens:", len(corpus_tokens))
print("Vocabulary size:", len(vocabulary))
print("First 15 vocabulary entries:", vocabulary[:15])

## 7. Inspect the special-token IDs

In [ ]:
pad_token_id = token_to_id["[PAD]"]
unknown_token_id = token_to_id["[UNK]"]
begin_token_id = token_to_id["[BOS]"]
end_token_id = token_to_id["[EOS]"]

print("PAD:", pad_token_id)
print("UNK:", unknown_token_id)
print("BOS:", begin_token_id)
print("EOS:", end_token_id)

## 8. Encode text

Encoding normalizes and tokenizes the input, handles unknown tokens, adds sequence markers, and optionally truncates, pads, or returns a PyTorch tensor.

In [ ]:
def encode(
    text: str,
    *,
    max_length: int | None = None,
    truncation: bool = False,
    padding: bool = False,
    return_tensors: bool = False,
):
    token_ids = [
        token_to_id.get(token, unknown_token_id)
        for token in tokenize(text)
    ]

    token_ids = [
        begin_token_id,
        *token_ids,
        end_token_id,
    ]

    if max_length is not None and truncation:
        if max_length < 2:
            raise ValueError(
                "max_length must be at least 2 "
                "when special tokens are included."
            )

        token_ids = token_ids[:max_length]
        token_ids[-1] = end_token_id

    if max_length is not None and padding:
        if len(token_ids) > max_length:
            raise ValueError(
                "Encoded sequence is longer than "
                "max_length. Enable truncation."
            )

        token_ids = token_ids + [
            pad_token_id
        ] * (max_length - len(token_ids))

    if return_tensors:
        return torch.tensor(
            [token_ids],
            dtype=torch.long,
        )

    return token_ids


encoded_example = encode(
    "Alice entered a mysterious garden.",
    max_length=12,
    truncation=True,
    padding=True,
)

print(encoded_example)

## 9. Decode token IDs

The decoder can omit special tokens and repairs spacing before common punctuation marks.

In [ ]:
def decode(
    token_ids,
    *,
    skip_special_tokens: bool = False,
) -> str:
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().flatten().tolist()

    special_token_ids = {
        pad_token_id,
        unknown_token_id,
        begin_token_id,
        end_token_id,
    }

    tokens = []

    for token_id in token_ids:
        if skip_special_tokens and token_id in special_token_ids:
            continue

        tokens.append(
            id_to_token.get(
                int(token_id),
                "[UNK]",
            )
        )

    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    return text


print(decode(encoded_example))
print(
    decode(
        encoded_example,
        skip_special_tokens=True,
    )
)

## 10. Convert between tokens and IDs

In [ ]:
def convert_tokens_to_ids(
    tokens: list[str],
) -> list[int]:
    return [
        token_to_id.get(token, unknown_token_id)
        for token in tokens
    ]


def convert_ids_to_tokens(
    token_ids: list[int],
) -> list[str]:
    return [
        id_to_token.get(
            int(token_id),
            "[UNK]",
        )
        for token_id in token_ids
    ]


tokens = tokenize("Alice saw a white rabbit.")
ids = convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("IDs:", ids)
print("Recovered tokens:", convert_ids_to_tokens(ids))

## 11. Create a reusable tokenizer class

In [ ]:
class SimpleWordTokenizer:
    def __init__(self, training_text: str):
        self.special_tokens = [
            "[PAD]",
            "[UNK]",
            "[BOS]",
            "[EOS]",
        ]

        lexical_tokens = sorted(
            set(tokenize(training_text))
        )

        self.vocabulary = (
            self.special_tokens
            + [
                token
                for token in lexical_tokens
                if token not in self.special_tokens
            ]
        )

        self.token_to_id = {
            token: index
            for index, token in enumerate(
                self.vocabulary
            )
        }

        self.id_to_token = {
            index: token
            for token, index
            in self.token_to_id.items()
        }

        self.pad_token_id = self.token_to_id["[PAD]"]
        self.unknown_token_id = self.token_to_id["[UNK]"]
        self.begin_token_id = self.token_to_id["[BOS]"]
        self.end_token_id = self.token_to_id["[EOS]"]

    def tokenize(self, text: str) -> list[str]:
        return tokenize(text)

    def encode(
        self,
        text: str,
        *,
        max_length: int | None = None,
        truncation: bool = False,
        padding: bool = False,
        return_tensors: bool = False,
    ):
        token_ids = [
            self.token_to_id.get(
                token,
                self.unknown_token_id,
            )
            for token in self.tokenize(text)
        ]

        token_ids = [
            self.begin_token_id,
            *token_ids,
            self.end_token_id,
        ]

        if max_length is not None and truncation:
            if max_length < 2:
                raise ValueError(
                    "max_length must be at least 2."
                )

            token_ids = token_ids[:max_length]
            token_ids[-1] = self.end_token_id

        if max_length is not None and padding:
            if len(token_ids) > max_length:
                raise ValueError(
                    "Sequence exceeds max_length. "
                    "Enable truncation."
                )

            token_ids.extend(
                [self.pad_token_id]
                * (max_length - len(token_ids))
            )

        if return_tensors:
            return torch.tensor(
                [token_ids],
                dtype=torch.long,
            )

        return token_ids

    def decode(
        self,
        token_ids,
        *,
        skip_special_tokens: bool = False,
    ) -> str:
        if isinstance(token_ids, torch.Tensor):
            token_ids = (
                token_ids.detach()
                .cpu()
                .flatten()
                .tolist()
            )

        special_token_ids = {
            self.pad_token_id,
            self.unknown_token_id,
            self.begin_token_id,
            self.end_token_id,
        }

        tokens = []

        for token_id in token_ids:
            token_id = int(token_id)

            if (
                skip_special_tokens
                and token_id in special_token_ids
            ):
                continue

            tokens.append(
                self.id_to_token.get(
                    token_id,
                    "[UNK]",
                )
            )

        text = " ".join(tokens)
        return re.sub(
            r"\s+([.,!?;:])",
            r"\1",
            text,
        )

    def convert_tokens_to_ids(
        self,
        tokens: list[str],
    ) -> list[int]:
        return [
            self.token_to_id.get(
                token,
                self.unknown_token_id,
            )
            for token in tokens
        ]

    def convert_ids_to_tokens(
        self,
        token_ids: list[int],
    ) -> list[str]:
        return [
            self.id_to_token.get(
                int(token_id),
                "[UNK]",
            )
            for token_id in token_ids
        ]

    def __len__(self) -> int:
        return len(self.vocabulary)

## 12. Test the tokenizer class

In [ ]:
tokenizer = SimpleWordTokenizer(cleaned_text)

sample_text = "Alice followed the rabbit into a strange place."

encoded = tokenizer.encode(
    sample_text,
    max_length=16,
    truncation=True,
    padding=True,
    return_tensors=True,
)

decoded = tokenizer.decode(
    encoded,
    skip_special_tokens=True,
)

print("Vocabulary size:", len(tokenizer))
print("Encoded shape:", encoded.shape)
print("Encoded IDs:", encoded)
print("Decoded text:", decoded)

## Limitations

This teaching tokenizer is intentionally simple:

- it uses a fixed word-level vocabulary;
- unseen words become `[UNK]`;
- it does not learn subword units;
- punctuation handling is limited;
- it is designed primarily for English text;
- it does not serialize its vocabulary to disk;
- it does not implement batching masks or token-type IDs.

Production NLP systems commonly use subword tokenizers such as WordPiece, Byte-Pair Encoding, or Unigram tokenization.

## Exercises

1. Add an attention mask for padded sequences.
2. Save and reload the vocabulary as JSON.
3. Compare word-level and character-level vocabulary sizes.
4. Count the most frequent corpus tokens.
5. Add a minimum-frequency threshold.
6. Implement batch encoding.
7. Compare this tokenizer with a Hugging Face tokenizer.
8. Add support for preserving Unicode characters.